# MLP Activation Analysis

Deep analysis of MLP neuron activations: distributions, dead/saturated neurons,
neuron-token correlations, sparsity profiles, and neuron-logit attribution.

## Why This Matters

MLP activation provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_activation_analysis import (
    mlp_activation_distribution,
    dead_neuron_analysis,
    neuron_token_correlation,
    activation_sparsity_profile,
    neuron_logit_attribution,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Activation Distribution

In [ ]:
result = mlp_activation_distribution(model, tokens)
for i in range(cfg.n_layers):
    print(f"Layer {i}: mean={result['mean_activation'][i]:.4f}, "
          f"std={result['std_activation'][i]:.4f}, "
          f"max={result['max_activation'][i]:.4f}, "
          f"sparsity={result['sparsity'][i]:.1%}")

## Dead Neuron Analysis

In [ ]:
tokens_list = [tokens, jnp.array([1, 2, 3, 4, 5, 6, 7]), jnp.array([10, 20, 30, 40, 50, 60, 70])]
result = dead_neuron_analysis(model, tokens_list)
print(f"Total dead: {result['total_dead']} / {result['total_neurons']}")
for i in range(cfg.n_layers):
    print(f"  Layer {i}: {result['dead_fraction'][i]:.1%} dead")

## Neuron-Token Correlation

In [ ]:
for layer in range(cfg.n_layers):
    result = neuron_token_correlation(model, tokens, layer=layer, top_k=3)
    print(f"\nLayer {layer} top neurons:")
    for idx, max_act, max_pos in result['top_neurons']:
        print(f"  Neuron {idx}: max_act={max_act:.4f} at position {max_pos}")

## Activation Sparsity Profile

In [ ]:
result = activation_sparsity_profile(model, tokens)
for i in range(cfg.n_layers):
    print(f"Layer {i}: sparsity={result['layer_sparsity'][i]:.1%}, "
          f"active={result['mean_active_neurons'][i]:.0f}, "
          f"eff_width={result['effective_width'][i]:.1f}")

## Neuron-Logit Attribution

In [ ]:
result = neuron_logit_attribution(model, tokens, layer=0, top_k=5)
print(f"Top neuron-token pairs:")
for n, t, logit in result['top_neuron_token_pairs'][:10]:
    print(f"  Neuron {n} -> Token {t}: {logit:+.4f}")